<a href="https://colab.research.google.com/github/SleepyEveryD/NLP/blob/mathonly/notebooks/03_live_play.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 · Live play — the REAL test  (the game API)

**Who Wants to Be a PoliMillionaire?** — here the actual game we play, not our dev set.

The `live`-mode counterpart of notebook 01, this is:
- **01 = OFFLINE** ('our own test'): the hand-crafted dev set, gold known, accuracy computed locally.
- **03 = LIVE** ('the real test'): the game server feeds the questions and grades them; `correct`
  only **after** submitting we learn (from `AnswerResult`).

Same pipeline, same `EvalRecord`/JSONL log — only `config.mode='live'` and a logged-in `GameClient` differ.
The single switch is `run_session(pipeline, config, game_client=...)` (D-015).

> ⚠️ **This plays a REAL game.** A leaderboard attempt it consumes and the **30s/question timer it starts**.
> The leaderboard resets ~1 week before the deadline → save serious runs for then. Run the play cell
> deliberately you should — not by a stray Shift+Enter.

> **GPU needed.** Runtime ▸ Change runtime type ▸ T4 GPU, select you must.

## 1 · Setup — clone the repo, add `millionaire_client` to the path, install deps
The same clone as notebook 01, plus the course's `millionaire_client` package onto `sys.path` we add
(in the repo at `NLP_assignment_api_client/` it lives).

In [12]:
# Auto-reload edited src modules on every cell run -- so after a `git pull` the newest code lands without a
# manual importlib.reload or a restart. (Re-run the cell that USES the code, e.g. code-wire.)
# Colab's IPython ships an autoreload that does `from imp import reload`, and `imp` is GONE in Python 3.12 --
# so a tiny `imp` shim (reload only) we install first, then load the extension. BEST-EFFORT: any failure
# caught, so the cell never stalls (fall back: after a src pull, Runtime > Restart to pick changes up).
try:
    import sys as _sys, types as _types, importlib as _importlib
    if 'imp' not in _sys.modules:
        _imp = _types.ModuleType('imp')
        _imp.reload = _importlib.reload          # the one thing the old autoreload.py wants from `imp`.
        _sys.modules['imp'] = _imp
    _ip = get_ipython()
    _ip.run_line_magic('load_ext', 'autoreload')
    _ip.run_line_magic('autoreload', '2')
    print('autoreload: ON (src edits hot-reload on cell re-run)')
except Exception as _e:
    print(f'autoreload OFF ({type(_e).__name__}: {_e}) -- after a src pull, Runtime > Restart to pick changes up.')

import os, sys

REPO_URL = 'https://github.com/SleepyEveryD/NLP.git'
REPO_ROOT = '/content/NLP'
BRANCH = 'mathonly'
if not os.path.exists(REPO_ROOT):
  !git clone -b {BRANCH} {REPO_URL} {REPO_ROOT}
else:
  # Already cloned -> HARD-SYNC to the latest pushed 4-rag (fetch + force-reset to origin/{BRANCH}).
  # Re-run this cell and the newest src ALWAYS lands (the News/calculator fix etc.) -- no stale checkout,
  # no "local changes block the pull". Tracked files are overwritten to match remote; UNTRACKED run
  # outputs are KEPT (experiments/runs/* is gitignored; wrong_questions.jsonl is untracked).
  # NOTE: `git pull` updates the FILES on disk -- it does NOT refresh THIS notebook's cells in the Colab
  #       browser. Code changes under src/ apply on the next import; notebook-cell edits need a re-open.
  !cd {REPO_ROOT} && git fetch -q origin && git checkout -q -f -B {BRANCH} origin/{BRANCH}

# Confirm the branch + the EXACT synced commit (so "did my fix land?" you can verify at a glance).
!cd {REPO_ROOT} && echo "on branch:" $(git rev-parse --abbrev-ref HEAD) "@" $(git --no-pager log -1 --oneline)
!grep -q "def default_tools" {REPO_ROOT}/src/tools/__init__.py && echo "✅ default_tools present (phase-3 src)" || echo "❌ still old src"

# Our src tree AND the provided client package, onto the import path both go.
SRC = os.path.join(REPO_ROOT, 'src')
API_CLIENT = os.path.join(REPO_ROOT, 'NLP_assignment_api_client')
for p in (SRC, API_CLIENT):
  if p not in sys.path:
    sys.path.insert(0, p)
# Into the repo root, change directory we do -- relative paths simpler they become.
os.chdir(REPO_ROOT)
print('Repo root:', REPO_ROOT)
print('On sys.path:', SRC, '|', API_CLIENT)

# The provided client, importable it must be -- confirm we do, before any game touch.
from millionaire_client import MillionaireClient
print('millionaire_client, imported it is.')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
autoreload: ON (src edits hot-reload on cell re-run)
on branch: mathonly @ 94ec6a6 。
✅ default_tools present (phase-3 src)
Repo root: /content/NLP
On sys.path: /content/NLP/src | /content/NLP/NLP_assignment_api_client
millionaire_client, imported it is.


In [13]:
# The inference stack + the client's `requests`, install we do (light it stays).
# NOTE: `-U` we MUST keep -- Colab a stale bitsandbytes (<0.46.1) preinstalls, and a bare
# `>=0.43.0` pip leaves it be (already satisfied), so the 4-bit loader then ImportErrors.
# `>=0.46.1` pinned + `-U` => the upgrade actually happens.
#
# ⚠️ pandas / requests, PINNED to Colab's own versions they are -- bare names + `-U` grabbed
# pandas 3.0.x & requests 2.34, which break `google-colab` (==2.2.2 / ==2.32.4) AND cudf / gradio /
# bqplot / db-dtypes (all need pandas<3). Matching Colab exactly -> zero dependency conflicts, and the
# pandas-3 breaking changes (copy-on-write, dropped APIs) never touch our metrics tables.
!pip install -q -U 'transformers>=4.45.0' 'accelerate>=0.34.0' 'bitsandbytes>=0.46.1' sentencepiece einops pyyaml 'pandas==2.2.2' matplotlib 'requests==2.32.4'
print('Installed, the dependencies are.')

Installed, the dependencies are.


In [14]:
import torch

# A GPU, present it must be -- on CPU, a 7B model in 30s answer we cannot.
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING -- no GPU. Runtime ▸ Change runtime type ▸ T4 GPU, switch you must.')

CUDA available: True
GPU: Tesla T4


## 2 · Load the LIVE run config
`configs/live.yaml` carries `mode: 'live'`. The competition + game mode, here you choose.

Competitions: 0 Entertainment · 1 History · 2 Science · 3 Maths · 4 Philosophy · 5 News.

In [15]:
from config import RunConfig

# The live config, from YAML we load.
config = RunConfig.from_yaml(os.path.join(REPO_ROOT, 'configs', 'live.yaml'))

# Which competition to play + how, here choose it you do.
config.game.competition_id = 0          # 0..5
config.game.game_mode = 'text'          # 'text' | 'speech'
config.run_id = f'live_comp{config.game.competition_id}'

print('mode:', config.mode)
print('competition_id:', config.game.competition_id, '| game_mode:', config.game.game_mode)
print('aim_seconds:', config.game.aim_seconds, '(below the 30s wall, a network margin this keeps)')
print('model:', config.model.name, '|', config.model.quantization)

mode: live
competition_id: 0 | game_mode: text
aim_seconds: 25.0 (below the 30s wall, a network margin this keeps)
model: Qwen/Qwen2.5-7B-Instruct | 4bit


## 3 · Load + warm up the model
Identical to notebook 01 — the cold-start cost paid **before** any timed question, so the 30s wall a cold load never eats.

In [16]:
import time
from inference.engine import TransformersEngine

# Once, the model we load -- the cold-start cost, here we pay it.
t0 = time.perf_counter()
if 'engine' not in globals():
      engine = TransformersEngine(model_name=config.model.name,
                                  quantization=config.model.quantization,
                                  dtype=config.model.dtype)
      engine.warmup()
else:
      print('engine 已在显存中,跳过加载。')
print(f'Model loaded in {time.perf_counter() - t0:.1f}s')

# Warm up -- the first-call kernels, compiled before any timed question they are.
t0 = time.perf_counter()
engine.warmup()
print(f'Warmup in {time.perf_counter() - t0:.1f}s')

engine 已在显存中,跳过加载。
Model loaded in 0.0s
Warmup in 0.6s


## 4 · Wire the pipeline (same as offline) + log in to the game
DI exactly as notebook 01 (D-006) — only now a logged-in `GameClient` we also build.

Credentials: a PoliMi email as the username; the password from a **Colab secret** named
`poli-millionaire` we read (hardcode it, we must NOT — B3). Add it via the 🔑 panel on the left.

In [ ]:
from classify.classifier import QuestionClassifier
from prompting.builder import PromptBuilder
from agent.pipeline import QAPipeline
from tools import default_tools
from retrieval import build_retriever

# Phase 4 RAG: the routing retriever, built from config ONLY when config.retrieval.enabled.
# Ablation (RAG on vs off) = the `enabled` flag. `source` picks the strategy:
#   "routed"    -> per question: News -> live web (DuckDuckGo), else -> FAISS corpus (or Wikipedia).
#   "wikipedia" | "web" | "faiss" -> single-backend ablations.
# `needs_retrieval` still gates per question. News (post-cutoff) the live web NEEDS -- Wikipedia alone
# left News at 2/7 last sweep, the breaking 2026 facts it cannot hold; DuckDuckGo for those we add.
retriever = build_retriever(config.retrieval)
print('config.retrieval:', config.retrieval)   # the LOADED flags -- if RAG is OFF, here enabled=False you see.
print('RAG:', (f'ON  source={config.retrieval.source}  top_k={config.retrieval.top_k}') if retriever else 'OFF')

# The collaborators, injected into the pipeline they are (D-006) -- identical to offline.
pipeline = QAPipeline(
    engine=engine,
    prompt_builder=PromptBuilder(strategy=config.prompt_strategy),
    classifier=QuestionClassifier(),
    retriever=retriever,       # Phase 4: RAW evidence; News->web, else->FAISS/Wikipedia; gated + graceful.
    tools=default_tools(),     # Phase 3 ON: the safe-AST calculator. ONLY on maths questions it fires
                               # (needs_calculator gates it) -- on Entertainment etc. a harmless no-op it is.
    latency_budget_s=config.latency_budget_s,
)

# --- Maths (comp 3): REVERTED to the 5bcf593 known-good config (Maths 9/10, reached level 9) ---
# After 5bcf593 we tried two "improvements" that both made Maths worse:
#   * 403b989 added the calculator as a match-gated verifier -- it helped on some Qs but introduced new
#     failure modes (e.g. Q6767 group-theory: calc emitted 4*12/(2*2)=12 mapping to wrong option).
#   * 5130bac switched to cot_maths_v1 with worked exemplars -- the exemplars anchored variable setup
#     (Q6777 chain now writes "Let s=speed, p=price") but the model still slipped on the algebra step
#     and answer-to-option mapping; net result was no better than cot_v2.
# Reverting to the cot_v2 + NO-calculator + single-pass config until we have evidence a change wins.
# cot_maths_v1 stays REGISTERED (no harm) but unused; the calculator stays in src but Maths skips it.
pipeline_maths = QAPipeline(
    engine=engine,
    prompt_builder=PromptBuilder(strategy='cot_v2'),   # cot_v1 + option-matching check (run #7 fix).
    classifier=QuestionClassifier(),
    retriever=None,            # Maths: NO retrieval -- it only distracts here.
    tools=None,                # NO calculator -- at n=1 it would clobber cot_v2 on numeric Qs (run #8).
    latency_budget_s=config.latency_budget_s,
    # self_consistency_n defaults to 1 -- SC dropped for Maths (run #8: it timed out, gave no benefit).
)
print('Maths pipeline (comp 3): cot_v2 + single-pass (n=1) + NO retrieval + NO calculator')

# --- Log in to the real game ---
from google.colab import userdata
from game.client import GameClient

USERNAME = userdata.get('username')       # <-- your PoliMi email, fill it you must.
PASSWORD = userdata.get('password')  # <-- a Colab secret, NOT hardcoded (B3).

game_client = GameClient()
game_client.login(USERNAME, PASSWORD)
print('Logged in as', USERNAME)

# The competitions and their ids, list them we do (safe -- no timer this starts).
for c in game_client.list_competitions():
    ml = getattr(c, 'max_levels', '?')
    print('  id=', c.id, '|', c.name, '| max_levels=', ml)

## 5 · ▶ Play ONE real game  (consumes a leaderboard attempt + starts the 30s timer)
`run_session` sees `mode='live'` and drives `LiveRunner`: start game → our pipeline answers each
question → submit by integer `Option.id` → log one `EvalRecord` per turn (with `correct` from the server).

⚠️ Deliberately run this. Each turn prints live; the full per-turn record lands in the JSONL log.

In [18]:
from evaluation.runner import run_session

# The ONE switch -- mode='live' it sees, so LiveRunner over the real game it drives.
run_path = run_session(
    pipeline,
    config,
    game_client=game_client,
    log_root=os.path.join(REPO_ROOT, 'experiments', 'runs'),
)
print('Live run written to:', run_path)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1] qid=128 lvl=0 reached=0 -> B | correct=False | timed_out=False | latency=2.0s (left was 29.907317)
Live run written to: /content/NLP/experiments/runs/live_comp0


## 6 · Results — how far did we get?
From the same JSONL log as offline it derives. `correct` is None where the server withheld it
(e.g. a timeout). Highest level reached + a per-turn table, here we show.

In [19]:
from evaluation.metrics import load_runs

# The live run, back into a DataFrame we read.
df = load_runs([run_path])


def _run_reached(df):
    """The level THIS run CLIMBED to -- the server's `reached_level` telemetry (from AnswerResult),
    its max across the turns. NOT `level` (that is the per-turn rung the server sends as 0 -- the old
    bug, where 'Highest level reached: 0' it always printed). Fallbacks: current_level, then level.
    None, only when the server told us nothing at all."""
    for col in ('reached_level', 'current_level', 'level'):
        if col in df.columns:
            s = df[col].dropna()
            if not s.empty:
                return int(s.max())
    return None


answered = len(df)
known = df[df['correct'].notna()]
n_correct = int(known['correct'].astype(float).sum()) if len(known) else 0
acc = (n_correct / len(known)) if len(known) else float('nan')
reached = _run_reached(df)   # FIXED: the server's reached_level, not the always-0 per-turn `level`.

print(f'Questions answered: {answered}')
print(f'Correct: {n_correct}/{len(known)}  (accuracy {acc:.0%})')
print(f'Highest level reached: {reached}')

# Per question, the full card -- the choices too, so the wrong ones inspect we can
# (the picked letter, by its option text now we read). Note: `options` only the live
# runs from NOW ON carry; an older log an empty cell shows.
for _, row in df.iterrows():
    opts = row['options'] if 'options' in df.columns and isinstance(row['options'], dict) else {}
    picked = row['predicted_answer']
    mark = {True: 'CORRECT', False: 'WRONG'}.get(row['correct'], 'UNKNOWN')  # None (server withheld) it is.
    print('\n' + '=' * 70)
    print(f"[{mark}] qid={row['qid']} | reached_level={row.get('reached_level')} | topic={row['topic']} | latency={row['latency_s']:.1f}s")
    print(f"Q: {row['question_text']}")
    if opts:
        for letter, text in opts.items():
            arrow = '   <-- model picked' if letter == picked else ''
            print(f"   {letter}. {text}{arrow}")
    else:
        print(f"   (options not logged for this run)  model picked: {picked}")
    print(f"correct={row['correct']}  |  confidence={row['confidence']}")


Questions answered: 1
Correct: 0/1  (accuracy 0%)
Highest level reached: 0

[WRONG] qid=128 | reached_level=0 | topic=None | latency=2.0s
Q: How does the film's setting, Isla Nublar, differ from the real-world locations considered for filming?
   A. Isla Nublar was filmed in Costa Rica
   B. Isla Nublar was filmed in Canada   <-- model picked
   C. Isla Nublar was filmed in Mexico
   D. Isla Nublar was filmed in Hawaii
correct=False  |  confidence=1.0


## 7 · Observations + next steps

_Fill in after the first real game (feeds the investigation rubric + confirms the API contract):_

- **API contract check (Phase-0 TODO):** did `Option.id` submission + `time_remaining` behave as documented?
- **Latency under the real wall:** any turn near 30s once network RTT is counted? Tune `game.aim_seconds`.
- **Where we fall:** the level + topic of the first wrong answer — a knowledge gap, or a parse slip?
- **Offline vs live gap:** does live accuracy track the ~87% the dev set predicted, or drift?

**Switching modes is a one-liner:** offline ⇄ live is just `config.mode` + whether you pass `game_client`.
For a quick offline re-check: `run_session(pipeline, RunConfig.from_yaml('configs/base.yaml'))` — no game_client.

> Be polite to the server (rate limits are real). Save real leaderboard pushes for after the ~1-week-pre-deadline reset.

## 8 · ▶ Sweep ALL competitions (live) + scores + every wrong question

Plays ONE live game in **each** of the 6 competitions (0–5), back to back, with a polite pause between
(the assignment asks: avoid rapid consecutive requests). Each competition logs its own run dir
(`live_comp{id}`). Then a per-competition **scoreboard** + the consolidated list of **every wrong
question** — text, options, and the letter our model picked — also saved to `experiments/wrong_questions.jsonl`.

> ⚠️ This plays **6 REAL games** (timers + leaderboard). Run it deliberately. **Maths (comp 3)** runs its
> own pipeline — `cot_v2` (CoT + option-matching check), **single-pass** (self-consistency was dropped after
> run #8 timed out at 41s), no retrieval, no calculator; every other competition uses the shared `few_shot_v1`
> pipeline. **RAG (Phase 4) is ON** when `configs/live.yaml` has `retrieval.enabled: true` —
> `source: "routed"` sends **News → live web (DuckDuckGo)** and every other topic → **FAISS corpus /
> Wikipedia**. (`needs_retrieval` still gates it per question.) One game failing (e.g. a rate-limit) won't
> abort the rest — it's logged as a blank row and the sweep continues.

This is the multi-competition counterpart of sections 5–6 (which play a single chosen competition).


In [20]:
from evaluation.runner import run_all_competitions

# Clear THIS sweep's prior run dirs FIRST. The logger appends within a run and the dir name is reused
# (live_comp{id}), so without this a re-run piles onto the previous one -> duplicate qids, inflated counts,
# mixed strategies. This is a SHELL command (not cached Python), so it works even if the logger's
# truncate-on-open fix hasn't loaded yet (that needs a kernel Restart; this rm does not).
!rm -rf {REPO_ROOT}/experiments/runs/live_comp*
print('cleared prior live_comp* run dirs')

COMP_NAMES = {0: 'Entertainment', 1: 'Ancient History & Politics', 2: 'Science & Nature',
              3: 'Maths', 4: 'Philosophy & Psychology', 5: 'News'}

# One live game in EVERY competition (0-5), back to back, a polite pause between (PDF: no rapid requests).
# Each competition its own run dir gets (live_comp{id}); one game's failure the rest never sinks.
# Maths (comp 3) its OWN pipeline gets via `pipeline_for` -- cot_v2 + single-pass (n=1) + NO retrieval + NO
# calculator (run #8: cot_v2 + self-consistency timed out at 41s on a question it answered CORRECTLY; SC gave
# no benefit, so dropped). Routed by competition_id, the reliable LIVE signal; every other competition the
# shared few_shot + RAG `pipeline` keeps.
comp_runs = run_all_competitions(
    pipeline, config, game_client,
    competition_ids=range(6),
    log_root=os.path.join(REPO_ROOT, 'experiments', 'runs'),
    pause_s=8.0,
    on_competition=lambda cid: print(f"\n===== ▶ Competition {cid}: {COMP_NAMES.get(cid, '?')} ====="),
    pipeline_for=lambda cid: pipeline_maths if cid == 3 else pipeline,
)
print('\nSweep done. Run paths:')
for cid, path in comp_runs:
    print(f"  comp {cid} {COMP_NAMES.get(cid, ''):28} -> {path}")

cleared prior live_comp* run dirs

===== ▶ Competition 0: Entertainment =====


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1] qid=728 lvl=0 reached=None -> B | correct=True | timed_out=False | latency=1.9s (left was 29.908854)
[2] qid=233 lvl=0 reached=None -> B | correct=True | timed_out=False | latency=2.6s (left was 29.909972)
[3] qid=44 lvl=0 reached=None -> B | correct=True | timed_out=False | latency=5.8s (left was 29.910269)
[4] qid=510 lvl=0 reached=3 -> B | correct=False | timed_out=False | latency=5.3s (left was 29.908075)

===== ▶ Competition 1: Ancient History & Politics =====


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1] qid=1290 lvl=0 reached=0 -> D | correct=False | timed_out=False | latency=5.4s (left was 29.910751)

===== ▶ Competition 2: Science & Nature =====


...

[1] qid=6787 lvl=0 reached=0 -> C | correct=False | timed_out=False | latency=8.4s (left was 29.910036)

===== ▶ Competition 4: Philosophy & Psychology =====


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1] qid=7197 lvl=0 reached=None -> C | correct=True | timed_out=False | latency=1.9s (left was 29.910385)
[2] qid=7847 lvl=0 reached=None -> C | correct=True | timed_out=False | latency=1.8s (left was 29.910075)
[3] qid=9835 lvl=0 reached=None -> C | correct=True | timed_out=False | latency=1.8s (left was 29.908137)
[4] qid=7926 lvl=0 reached=None -> C | correct=True | timed_out=False | latency=1.8s (left was 29.90775)
[5] qid=8002 lvl=0 reached=None -> B | correct=True | timed_out=False | latency=1.8s (left was 29.910129)
[6] qid=8118 lvl=0 reached=None -> C | correct=True | timed_out=False | latency=1.9s (left was 29.908534)
[7] qid=7488 lvl=0 reached=None -> C | correct=True | timed_out=False | latency=1.9s (left was 29.908585)
[8] qid=8685 lvl=0 reached=None -> A | correct=True | timed_out=False | latency=1.9s (left was 29.907065)
[9] qid=7347 lvl=0 reached=None -> C | correct=True | timed_out=False | latency=2.0s (left was 29.908068)
[10] qid=8798 lvl=0 reached=None -> D | correct

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1] qid=11386 lvl=0 reached=None -> A | correct=True | timed_out=False | latency=2.3s (left was 29.910417)
[2] qid=10884 lvl=0 reached=1 -> C | correct=False | timed_out=False | latency=4.4s (left was 29.908225)

Sweep done. Run paths:
  comp 0 Entertainment                -> /content/NLP/experiments/runs/live_comp0
  comp 1 Ancient History & Politics   -> /content/NLP/experiments/runs/live_comp1
  comp 2 Science & Nature             -> /content/NLP/experiments/runs/live_comp2
  comp 3 Maths                        -> /content/NLP/experiments/runs/live_comp3
  comp 4 Philosophy & Psychology      -> /content/NLP/experiments/runs/live_comp4
  comp 5 News                         -> /content/NLP/experiments/runs/live_comp5


In [21]:
import json
import pandas as pd
from evaluation.metrics import load_runs


def _retr_docs(r):
    """The retrieved doc ids as a clean list, defensively we read (NaN/None -> [])."""
    v = r.get('retrieved_doc_ids')
    return list(v) if isinstance(v, (list, tuple)) else []


def _run_reached(df):
    """The level THIS run CLIMBED to -- the server's `reached_level` telemetry (from AnswerResult),
    its max across the turns. NOT `level` (the per-turn rung the server sends as 0 -- the old bug that
    made this column always 0). Fallbacks: current_level, then level. None only if the server said nothing."""
    for col in ('reached_level', 'current_level', 'level'):
        if col in df.columns:
            s = df[col].dropna()
            if not s.empty:
                return int(s.max())
    return None


def _lb_entry(cid):
    """The AUTHORITATIVE leaderboard row for us in this competition (reached_level + score) -- the REAL
    scored metric (D-014). Crash-safe: None on any error (player not found, client shape differs, ...)."""
    try:
        me = game_client._client.user.username
        return game_client._client.leaderboard.find_player(cid, me)
    except Exception:
        return None


rows, wrong_all, usage = [], [], []
for cid, path in comp_runs:
    lb = _lb_entry(cid)
    lb_level = getattr(lb, 'reached_level', None)
    lb_score = getattr(lb, 'score', None)
    if path is None:   # That competition failed (e.g. rate-limited) -- a blank row it gets (lb still shown).
        rows.append({'comp': cid, 'name': COMP_NAMES.get(cid, ''), 'answered': 0,
                     'correct': 0, 'of': 0, 'accuracy': float('nan'),
                     'run_reached': None, 'lb_level': lb_level, 'lb_score': lb_score})
        continue
    df = load_runs([path])
    known = df[df['correct'].notna()]
    n_correct = int(known['correct'].astype(float).sum()) if len(known) else 0
    acc = (n_correct / len(known)) if len(known) else float('nan')
    rows.append({'comp': cid, 'name': COMP_NAMES.get(cid, ''), 'answered': len(df),
                 'correct': n_correct, 'of': len(known), 'accuracy': acc,
                 # run_reached = THIS sweep's climb (server telemetry); lb_level = all-time scored best.
                 'run_reached': _run_reached(df), 'lb_level': lb_level, 'lb_score': lb_score})
    # Per-competition tool/retrieval usage -- the diagnostic the wrong-dump was MISSING.
    # retrieval_fired = needs_retrieval gated ON; docs_landed = snippets actually came back
    # (fired but 0 docs => the backend returned nothing, e.g. DuckDuckGo blocked on the Colab IP).
    n_retr = int(df['retrieval_used'].fillna(False).astype(bool).sum()) if 'retrieval_used' in df else 0
    n_docs = int(sum(len(_retr_docs(r)) > 0 for _, r in df.iterrows())) if 'retrieved_doc_ids' in df else 0
    n_tool = int(df['tool_used'].notna().sum()) if 'tool_used' in df else 0
    usage.append({'comp': cid, 'name': COMP_NAMES.get(cid, ''), 'answered': len(df),
                  'retrieval_fired': n_retr, 'docs_landed': n_docs, 'tool_calls': n_tool})
    for _, r in df[df['correct'] == False].iterrows():   # correct is None (timeout) -> excluded, good.
        wrong_all.append((cid, COMP_NAMES.get(cid, ''), r))

# --- Scoreboard, per competition + overall ---
# run_reached = how far THIS sweep climbed (server's reached_level telemetry, NOT the always-0 per-turn rung).
# lb_level / lb_score = the AUTHORITATIVE leaderboard numbers (all-time best -- what we are actually scored on).
summary = pd.DataFrame(rows)
print('SCORES BY COMPETITION')
print(summary.to_string(index=False))
tot_c, tot_n = int(summary['correct'].sum()), int(summary['of'].sum())
print((f"\nOVERALL: {tot_c}/{tot_n} = {tot_c / tot_n:.1%}") if tot_n else "\nOVERALL: no graded answers")

# --- RAG / tool usage by competition (the News path, here we finally SEE it) ---
print('\nRAG / TOOL USAGE BY COMPETITION')
print(pd.DataFrame(usage).to_string(index=False))

# --- Every wrong question: text + options + the letter our model picked + WHY (tools/retrieval) ---
print(f"\n{'=' * 72}\nEVERY WRONG QUESTION  ({len(wrong_all)})\n{'=' * 72}")
for cid, name, r in wrong_all:
    opts = r['options'] if 'options' in r and isinstance(r['options'], dict) else {}
    picked = r['predicted_answer']
    docs = _retr_docs(r)
    print(f"\n[comp {cid} · {name}] qid={r['qid']} reached_level={r.get('reached_level')}")
    print(f"Q: {r['question_text']}")
    for letter, text in opts.items():
        mark = '   <-- our pick (WRONG)' if letter == picked else ''
        print(f"   {letter}. {text}{mark}")
    # The diagnostic line -- did a tool fire? did retrieval fire, and did snippets LAND?
    # tool=None + retrieval_used=False  => the plain model answered (no help asked for).
    # retrieval_used=True + docs_landed=0 => the retriever fired but came back EMPTY (the News bug to watch).
    print(f"   -> tool={r.get('tool_used')} | retrieval_used={bool(r.get('retrieval_used'))} | "
          f"docs_landed={len(docs)}" + (f" {docs[:3]}" if docs else ""))

# --- Save a clean consolidated record of the wrong questions (now WITH the tool/retrieval trace) ---
out = os.path.join(REPO_ROOT, 'experiments', 'wrong_questions.jsonl')
with open(out, 'w', encoding='utf-8') as f:
    for cid, name, r in wrong_all:
        f.write(json.dumps({
            'competition_id': cid, 'competition': name, 'qid': r['qid'],
            'level': r['level'], 'reached_level': r.get('reached_level'),
            'question_text': r['question_text'],
            'options': r['options'] if isinstance(r.get('options'), dict) else {},
            'our_wrong_pick': r['predicted_answer'],
            'tool_used': r.get('tool_used'),
            'retrieval_used': bool(r.get('retrieval_used')),
            'retrieved_doc_ids': _retr_docs(r),
        }, ensure_ascii=False) + '\n')
print(f"\nWrong questions saved -> {out}  ({len(wrong_all)} rows)")


SCORES BY COMPETITION
 comp                       name  answered  correct  of  accuracy  run_reached  lb_level  lb_score
    0              Entertainment         4        3   4      0.75            3        15   1024000
    1 Ancient History & Politics         1        0   1      0.00            0        15   1024000
    2           Science & Nature        15       15  15      1.00           15        15   1024000
    3                      Maths         1        0   1      0.00            0         9     16000
    4    Philosophy & Psychology        15       15  15      1.00           15        15   1024000
    5                       News         2        1   2      0.50            1         5      1000

OVERALL: 34/38 = 89.5%

RAG / TOOL USAGE BY COMPETITION
 comp                       name  answered  retrieval_fired  docs_landed  tool_calls
    0              Entertainment         4                3            2           0
    1 Ancient History & Politics         1                

In [22]:
import json, collections
from pathlib import Path

# Per-question tool/retrieval/STRATEGY detail across ALL competitions. The diagnostics:
#   - retr=True + docs=0  => the retriever fired but came back EMPTY (DDG blocked / Wikipedia miss).
#   - strat=cot_v1 on comp 3 ONLY  => the Maths routing (pipeline_for) took effect; few_shot_v1 elsewhere.
# Logger is now truncate-per-run (one sweep = one fresh records.jsonl), so these rows are THIS run only.
for cid, path in comp_runs:
    if path is None:
        continue
    p = Path(path) / "records.jsonl"
    if not p.exists():
        continue
    rows = [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]
    print(f"\n===== comp {cid} {COMP_NAMES.get(cid, '')}  (n={len(rows)}) =====")
    print("  tool_used:      ", collections.Counter(r.get("tool_used") for r in rows))
    print("  retrieval_used: ", collections.Counter(r.get("retrieval_used") for r in rows))
    print("  prompt_strategy:", collections.Counter(r.get("prompt_strategy") for r in rows))
    for r in rows:
        docs = r.get("retrieved_doc_ids") or []
        flag = '' if r.get('correct') else '  <-- WRONG'
        print(f"  {r['qid']:>6} strat={str(r.get('prompt_strategy')):<12} "
              f"tool={str(r.get('tool_used')):<11} retr={str(r.get('retrieval_used')):<5} "
              f"docs={len(docs):<2} correct={str(r.get('correct')):<5} | {r['question_text'][:50]}{flag}")



===== comp 0 Entertainment  (n=4) =====
  tool_used:       Counter({None: 4})
  retrieval_used:  Counter({True: 3, False: 1})
  prompt_strategy: Counter({'few_shot_v1': 4})
     728 strat=few_shot_v1  tool=None        retr=False docs=0  correct=True  | Which of the following best describes the primary 
     233 strat=few_shot_v1  tool=None        retr=True  docs=0  correct=True  | How does Marlon Brando's portrayal of Stanley Kowa
      44 strat=few_shot_v1  tool=None        retr=True  docs=3  correct=True  | What is the connection between Humphrey Bogart's c
     510 strat=few_shot_v1  tool=None        retr=True  docs=3  correct=False | Which of the following best describes Clark Gable'  <-- WRONG

===== comp 1 Ancient History & Politics  (n=1) =====
  tool_used:       Counter({None: 1})
  retrieval_used:  Counter({True: 1})
  prompt_strategy: Counter({'few_shot_v1': 1})
    1290 strat=few_shot_v1  tool=None        retr=True  docs=3  correct=False | Which concept refers to the divisi